# ASHA Sahayak — GRPO Training

**Environment**: [ASHA Sahayak HF Space](https://sreenathmmenon-asha-sahayak.hf.space)  
**Algorithm**: GRPO via TRL  
**Efficiency**: Unsloth when GPU available, plain TRL as fallback  
**Base model**: Qwen/Qwen3-0.6B  
**Ground truth**: Indian Government IMNCI Protocol  

## Requirements
- **GPU runtime** (Runtime → Change runtime type → T4 GPU) — free Colab tier is sufficient
- HuggingFace token with write access (for pushing trained model to Hub)
- No premium subscription required for training

## What this notebook does
1. Installs Unsloth (GPU-accelerated) or plain TRL (CPU fallback)
2. Connects to live ASHA Sahayak environment
3. Runs GRPO training — model learns to ask right questions and make correct referrals
4. Plots real reward curves from actual training
5. Pushes trained model to HuggingFace Hub

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys, os, importlib

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# Core deps always needed
_pip('httpx>=0.24.0', 'datasets>=2.14.0', 'matplotlib', 'tqdm')

# Detect GPU
HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    pass

print(f'GPU available: {HAS_GPU}')

if HAS_GPU:
    # Exact pinned versions from official Unsloth Qwen3-GRPO notebook
    # vllm==0.15.1 required — newer vLLM breaks Unsloth's torch.compile graph
    # PYTORCH_ALLOC_CONF must be cleared — Colab sets expandable_segments:True by default
    # which conflicts with UNSLOTH_VLLM_STANDBY=1
    os.environ["PYTORCH_ALLOC_CONF"] = ""
    os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
    _pip('vllm==0.15.1')
    _pip('trl==0.22.2', 'transformers==4.56.2', 'accelerate', 'peft')
    _pip('unsloth', 'unsloth_zoo')
    # Refresh sys.path so newly installed packages are importable without restart
    importlib.invalidate_caches()
    import site
    if hasattr(site, 'getusersitepackages'):
        sp = site.getusersitepackages()
        if sp not in sys.path:
            sys.path.insert(0, sp)
    for sp in site.getsitepackages():
        if sp not in sys.path:
            sys.path.insert(0, sp)
    USE_UNSLOTH = True
    import unsloth
    print(f'Unsloth {unsloth.__version__} installed')
else:
    _pip('trl==0.22.2', 'transformers==4.56.2', 'accelerate', 'peft')
    USE_UNSLOTH = False
    print('No GPU — CPU fallback.')

print(f'Training backend: {"Unsloth + vLLM" if USE_UNSLOTH else "Plain TRL"}')
print('Cell 1 complete — run Cell 2 now.')

In [ ]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────
import os, torch

# Re-set env vars after runtime restart (restart wipes os.environ)
os.environ["PYTORCH_ALLOC_CONF"] = ""
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# Detect GPU and backend
HAS_GPU = torch.cuda.is_available()
USE_UNSLOTH = HAS_GPU  # Unsloth only on GPU

# ── HF Token: loaded from Colab Secrets — NEVER hardcode tokens ──────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

assert HF_TOKEN, "HF_TOKEN not found. Add it via Colab Secrets (left sidebar → 🔑)"

ENV_BASE_URL = 'https://sreenathmmenon-asha-sahayak.hf.space'
HF_REPO_ID   = 'sreenathmmenon/asha-sahayak-grpo'

BASE_MODEL       = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR       = 'asha-sahayak-grpo'
NUM_TRAIN_STEPS  = 200
LEARNING_RATE    = 5e-6
NUM_GENERATIONS  = 4
MAX_SEQ_LEN      = 2048

# Verify environment is reachable
import httpx
r = httpx.get(f'{ENV_BASE_URL}/health', timeout=30)
assert r.status_code == 200 and r.json()['status'] == 'healthy', f'Env not reachable: {r.text}'
meta = httpx.get(f'{ENV_BASE_URL}/metadata', timeout=30).json()
print(f'Environment: {ENV_BASE_URL}')
print(f'Cases: {meta["num_cases"]} | Tasks: {meta["tasks"]} | Concurrent: {meta["supports_concurrent_sessions"]}')
print(f'Base model: {BASE_MODEL}')
print(f'GPU: {torch.cuda.get_device_name(0) if HAS_GPU else "None"}')
print(f'Backend: {"Unsloth + vLLM" if USE_UNSLOTH else "Plain TRL"}')
print('HF Token: loaded ✓')

In [ ]:
# ── Cell 3: ASHA Environment client + TRL-compatible wrapper ─────────────────
from typing import Optional

class AshaClient:
    """Thin HTTP client for the ASHA Sahayak OpenEnv environment."""

    def __init__(self, base_url: str = ENV_BASE_URL):
        self.base_url = base_url.rstrip('/')
        self.session_id: Optional[str] = None

    def reset(self, task_id: str = 'easy', seed: int = 42) -> dict:
        resp = httpx.post(
            f'{self.base_url}/reset',
            json={'task_id': task_id, 'seed': seed},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        self.session_id = data['session_id']
        return data

    def step(self, *, referral_decision: str = 'PENDING', urgency: str = 'monitor',
             primary_concern: str = '', question: Optional[str] = None,
             confidence: float = 0.8) -> dict:
        headers = {'X-Session-ID': self.session_id} if self.session_id else {}
        resp = httpx.post(
            f'{self.base_url}/step',
            json={'referral_decision': referral_decision, 'urgency': urgency,
                  'primary_concern': primary_concern, 'question': question,
                  'confidence': confidence},
            headers=headers,
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()['observation']


class AshaToolEnv:
    """
    TRL-compatible RL environment for ASHA Sahayak.

    TRL discovers public methods as callable tools via their docstrings.
    reset() is called at episode start; reward is read from self.reward.
    """

    def __init__(self):
        self.client = AshaClient()
        self.reward: float = 0.0
        self._done: bool = False

    def reset(self, task_id: str = 'easy', seed: int = 42, **_) -> str:
        """Reset to a new clinical case. Called automatically by TRL."""
        self.reward = 0.0
        self._done = False
        data = self.client.reset(task_id=task_id, seed=seed)
        return data['observation']['conversation'][0]['text']

    def ask_question(self, question: str) -> str:
        """Ask the ASHA worker a clinical clarifying question.

        Args:
            question: Clinical question, e.g. 'Does the child have fast breathing?'

        Returns:
            ASHA worker's clinical observation in response to the question.
        """
        if self._done:
            return 'Episode already ended. No more questions.'
        obs = self.client.step(referral_decision='PENDING', question=question)
        self.reward = obs.get('reward', 0.0)
        self._done = obs.get('done', False)
        convo = obs.get('conversation', [])
        return convo[-1]['text'] if convo else 'No response.'

    def make_referral(
        self,
        referral_decision: str,
        urgency: str,
        primary_concern: str,
        confidence: float = 0.8,
    ) -> str:
        """Make the final clinical referral decision for the patient.

        Args:
            referral_decision: REFER_IMMEDIATELY | REFER_WITHIN_24H | TREAT_AT_HOME | MONITOR
            urgency: immediate | within_24h | routine | monitor
            primary_concern: Clinical condition, e.g. severe_pneumonia, pre_eclampsia
            confidence: Confidence score 0.0-1.0

        Returns:
            Clinical feedback with score breakdown.
        """
        if self._done:
            return 'Episode already ended.'
        obs = self.client.step(
            referral_decision=referral_decision, urgency=urgency,
            primary_concern=primary_concern, confidence=confidence,
        )
        self.reward = obs.get('reward', 0.0)
        self._done = obs.get('done', False)
        return obs.get('feedback') or f'Score: {self.reward:.3f}'


# Smoke test
env = AshaToolEnv()
obs = env.reset(task_id='easy', seed=42)
print('Case:', obs[:200])
resp = env.ask_question('Does the child have fast breathing or chest indrawing?')
print('Response:', resp[:150])
fb = env.make_referral('REFER_IMMEDIATELY', 'immediate', 'severe_pneumonia')
print('Feedback:', fb[:150])
print(f'Reward: {env.reward:.4f}')

In [ ]:
# ── Cell 4: Load model — official Unsloth GRPO pattern ─────────────────────
# Source: github.com/unslothai/notebooks — Openenv_wordle_grpo.ipynb and Qwen3_(4B)-GRPO.ipynb
# Key: load_in_4bit=False (16-bit LoRA), fast_inference=True (vLLM for generation)
import torch

COMPUTE_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float16
print(f'Compute dtype: {COMPUTE_DTYPE}')

if USE_UNSLOTH:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,          # 16-bit LoRA — no 4-bit quantization, no dtype mismatch
        fast_inference=True,         # vLLM for fast generation during GRPO rollouts
        max_lora_rank=64,
        gpu_memory_utilization=0.6,  # 0.6 required when UNSLOTH_VLLM_STANDBY=1 on Colab
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=64,
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    print('Model loaded: Unsloth 16-bit LoRA + vLLM inference')
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=COMPUTE_DTYPE,
        device_map='auto' if HAS_GPU else 'cpu',
        trust_remote_code=True,
    )
    lora_cfg = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.0, bias='none',
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_cfg)
    print('Model loaded: plain TRL 16-bit LoRA')

model.print_trainable_parameters()

In [ ]:
# ── Cell 5: Training dataset ─────────────────────────────────────────────────
import random
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI assistant helping ASHA (Accredited Social Health Activist) workers in rural India make clinical triage decisions using the IMNCI protocol.

Your workflow:
1. Ask clarifying questions with ask_question() to gather clinical information
2. Make your final decision with make_referral():
   - REFER_IMMEDIATELY: Life-threatening — call 108 now
   - REFER_WITHIN_24H: Needs PHC within 24 hours
   - TREAT_AT_HOME: Manage at home with ASHA guidance
   - MONITOR: Observe and follow up

Always ask at least one clarifying question before deciding.
Key danger signs: fast breathing, chest indrawing, inability to feed, convulsions, unconsciousness, severe bleeding, very low weight."""

USER_PROMPT = "A patient has been presented to you. Please assess and make a clinical triage decision."

random.seed(42)
rows = []

for seed in range(50):   rows.append({'task_id': 'easy',   'seed': seed})
for seed in range(100):  rows.append({'task_id': 'medium', 'seed': seed})
for seed in range(75):   rows.append({'task_id': 'hard',   'seed': seed})

random.shuffle(rows)

dataset = Dataset.from_list([
    {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': USER_PROMPT},
        ],
        'task_id': r['task_id'],
        'seed':    r['seed'],
    }
    for r in rows
])

counts = {t: sum(1 for r in rows if r['task_id'] == t) for t in ['easy', 'medium', 'hard']}
print(f'Dataset: {len(dataset)} episodes')
for t, n in counts.items():
    print(f'  {t}: {n}')

In [ ]:
# ── Cell 6: GRPO Training ─────────────────────────────────────────────────────
import os, re, torch, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
os.environ['TRL_EXPERIMENTAL_SILENCE'] = '1'

from trl import GRPOTrainer, GRPOConfig

REFERRALS = ["REFER_IMMEDIATELY", "REFER_WITHIN_24H", "TREAT_AT_HOME", "MONITOR"]
URGENCIES  = ["immediate", "within_24h", "routine", "monitor"]

def parse_and_score(text: str, task_id: str, seed: int) -> float:
    """Parse model output and score it against the ASHA environment."""
    text_upper = text.upper()
    referral = next((r for r in REFERRALS if r in text_upper), "MONITOR")
    urgency  = next((u for u in URGENCIES  if u in text.lower()), "monitor")

    # Extract primary concern
    concern_match = re.search(r'(?:concern|diagnosis|condition)[:\s]+([a-z_]+)', text.lower())
    concern = concern_match.group(1) if concern_match else "general"

    # Extract question if present
    question_match = re.search(r'(?:question|ask)[:\s]+"?([^"\n]{10,150})"?', text.lower())
    question = question_match.group(1) if question_match else None

    env = AshaToolEnv()
    env.reset(task_id=task_id, seed=seed)
    if question and not env._done:
        env.ask_question(question)
    if not env._done:
        env.make_referral(referral, urgency, concern)
    return float(env.reward)


def _score_one(i, completion, task_id_list, seed_list):
    """Score a single completion — called from thread pool."""
    try:
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)
        tid  = task_id_list[i] if task_id_list else 'easy'
        sd   = int(seed_list[i]) if seed_list else i
        return i, parse_and_score(text, tid, sd)
    except Exception as e:
        print(f"[reward] completion {i} failed: {e}")
        traceback.print_exc()
        return i, 0.0


def reward_func(prompts, completions, completion_ids, task_id=None, seed=None, **kwargs):
    n = len(completions)
    results = [0.0] * n
    with ThreadPoolExecutor(max_workers=min(n, 8)) as pool:
        futures = {pool.submit(_score_one, i, completions[i], task_id, seed): i for i in range(n)}
        for future in as_completed(futures):
            idx, reward = future.result()
            results[idx] = reward
    return results


# ── Pre-training smoke test ──────────────────────────────────────────────────
print("Smoke test: verifying reward scoring before training starts...")
_test_completions = [
    [{"role": "assistant", "content": "Question: Does the child have fast breathing? REFER_IMMEDIATELY urgency immediate concern severe_pneumonia"}],
    [{"role": "assistant", "content": "TREAT_AT_HOME urgency routine diagnosis pneumonia"}],
]
_test_rewards = reward_func(
    prompts=["dummy", "dummy"],
    completions=_test_completions,
    completion_ids=["dummy", "dummy"],
    task_id=["easy", "easy"],
    seed=[42, 42],
)
print(f"  Smoke test rewards: {[round(r,4) for r in _test_rewards]}")
assert any(r != 0.0 for r in _test_rewards), (
    "ABORT: All smoke test rewards are 0.0 — environment is not reachable or scoring is broken. "
    "Check Cell 2 ENV_BASE_URL and that the HF Space is running."
)
print("  Smoke test passed ✓  (rewards are non-zero)")
print()

# ── GRPOConfig ───────────────────────────────────────────────────────────────
# trl==0.22.2 — max_prompt_length is valid in this version
config = GRPOConfig(
    use_vllm=True,
    output_dir=OUTPUT_DIR,
    max_steps=NUM_TRAIN_STEPS,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_prompt_length=512,
    max_completion_length=512,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.1,
    logging_steps=1,
    save_steps=50,
    report_to='none',
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=reward_func,
)

print(f'Starting GRPO training ({NUM_TRAIN_STEPS} steps)...')
print(f'GPU: {torch.cuda.get_device_name(0) if HAS_GPU else "None"}')
trainer.train()
print('Training complete!')

In [ ]:
# ── Cell 7: Plot real training curves ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# TRL 1.2.x logs to trainer.state.log_history with key "reward"
# Each log entry looks like: {"step": N, "reward": 0.xxx, "reward_std": 0.xxx, ...}
logs = [l for l in trainer.state.log_history if 'reward' in l]
steps   = [l['step'] for l in logs]
rewards = [l['reward'] for l in logs]

if not rewards:
    print('No reward logs found.')
    print('Available keys in log_history[0]:', list(trainer.state.log_history[0].keys()) if trainer.state.log_history else 'empty')
    print('\nAll log entries:')
    for entry in trainer.state.log_history:
        print(' ', entry)
else:
    print(f'Found {len(rewards)} log points | Reward range: {min(rewards):.4f} – {max(rewards):.4f}')

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps, rewards, alpha=0.4, color='steelblue', linewidth=1, label='Per-step reward')

    # Smoothed curve
    if len(rewards) >= 5:
        w = max(3, len(rewards) // 10)
        smoothed = np.convolve(rewards, np.ones(w) / w, mode='valid')
        ax.plot(steps[w - 1:], smoothed, color='steelblue', linewidth=2.5, label='Smoothed')

    ax.set_xlabel('Training Step', fontsize=13)
    ax.set_ylabel('Episode Reward', fontsize=13)
    ax.set_title('ASHA Sahayak — GRPO Training (Qwen3-0.6B)', fontsize=15, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('asha_grpo_training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved: asha_grpo_training_curve.png')
    print(f'Final reward: {rewards[-1]:.4f} | Peak: {max(rewards):.4f}')
    if len(rewards) > 1:
        improvement = rewards[-1] - rewards[0]
        print(f'Improvement: {improvement:+.4f} ({improvement/max(abs(rewards[0]),1e-6)*100:.1f}%)')

In [ ]:
# ── Cell 8: Save and push to HuggingFace Hub ─────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to ./{OUTPUT_DIR}/')

if HF_TOKEN:
    if USE_UNSLOTH:
        # Unsloth: merge LoRA into base weights before pushing
        # Do NOT naively upcast 4-bit to 16-bit — use Unsloth's merge path
        model.save_pretrained_merged(
            OUTPUT_DIR + '_merged',
            tokenizer,
            save_method='merged_16bit',
        )
        model.push_to_hub_merged(
            HF_REPO_ID,
            tokenizer,
            save_method='merged_16bit',
            token=HF_TOKEN,
        )
    else:
        trainer.push_to_hub(repo_id=HF_REPO_ID, token=HF_TOKEN)
    print(f'Model pushed to: https://huggingface.co/{HF_REPO_ID}')
else:
    print('HF_TOKEN not set — model saved locally only.')
    print('To push: set HF_TOKEN and re-run this cell.')

In [ ]:
# ── Cell 9: Upload real training curve to HF Space ───────────────────────────
# Replaces the illustrative placeholder in assets/training_reward_curve.png
import os

if HF_TOKEN and os.path.exists('asha_grpo_training_curve.png'):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj='asha_grpo_training_curve.png',
        path_in_repo='assets/training_reward_curve.png',
        repo_id='sreenathmmenon/asha-sahayak',
        repo_type='space',
        commit_message='Update training curve with real GRPO results',
    )
    print('Real training curve uploaded to HF Space.')
    print('View at: https://sreenathmmenon-asha-sahayak.hf.space')
else:
    print('Skipped: either HF_TOKEN not set or plot file not found.')